# 02 | Validate and standardize Silver
Parse types, quarantine invalid records, apply referential checks, and keep the latest `updated_at` per order. The Silver layer is rebuilt from the complete Bronze history for clarity and reproducibility. This is **incremental ingestion, full-refresh Silver** rather than end-to-end incremental processing.

In [ ]:
from pyspark.sql import functions as F, Window
from pyspark.sql.types import DecimalType

for table in ["bronze_orders", "bronze_customers", "bronze_products"]:
    assert spark.catalog.tableExists(table), f"Run 01 first: {table} is missing"

customers = (spark.table("bronze_customers")
    .select(F.trim("customer_id").alias("customer_id"),
            F.trim("customer_name").alias("customer_name"),
            F.trim("region").alias("region"),
            F.to_date("signup_date").alias("signup_date"))
    .dropDuplicates(["customer_id"]))
products = (spark.table("bronze_products")
    .select(F.trim("product_id").alias("product_id"),
            F.trim("product_name").alias("product_name"),
            F.trim("category").alias("category"),
            F.col("list_price").cast(DecimalType(12,2)).alias("list_price"))
    .dropDuplicates(["product_id"]))

raw = spark.table("bronze_orders")
parsed = (raw.select(
    F.trim("order_id").alias("order_id"),
    F.expr("try_cast(order_date as date)").alias("order_date"),
    F.trim("customer_id").alias("customer_id"),
    F.trim("product_id").alias("product_id"),
    F.expr("try_cast(quantity as int)").alias("quantity"),
    F.expr("try_cast(unit_price as decimal(12,2))").alias("unit_price"),
    F.trim("status").alias("status"),
    F.expr("try_cast(updated_at as timestamp)").alias("updated_at"),
    "_source_file", "_ingested_at"))

# Prefer the newest source correction for a repeated order_id.
w = Window.partitionBy("order_id").orderBy(F.col("updated_at").desc_nulls_last(), F.col("_ingested_at").desc())
latest = parsed.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")
joined = (latest
    .join(customers.select("customer_id").withColumn("_has_customer", F.lit(1)), "customer_id", "left")
    .join(products.select("product_id").withColumn("_has_product", F.lit(1)), "product_id", "left"))
invalid = (F.col("order_id").isNull() | (F.length("order_id") == 0)
    | F.col("order_date").isNull() | F.col("updated_at").isNull()
    | F.col("customer_id").isNull() | (F.length("customer_id") == 0)
    | F.col("product_id").isNull() | (F.length("product_id") == 0)
    | F.col("quantity").isNull() | (F.col("quantity") <= 0)
    | F.col("unit_price").isNull() | (F.col("unit_price") <= 0)
    | ~F.col("status").isin("Completed", "Returned", "Cancelled")
    | F.col("_has_customer").isNull() | F.col("_has_product").isNull())
reason = (F.when(F.col("order_date").isNull(), "invalid_order_date")
    .when(F.col("updated_at").isNull(), "invalid_updated_at")
    .when(F.col("customer_id").isNull() | (F.length("customer_id") == 0) | F.col("_has_customer").isNull(), "invalid_customer")
    .when(F.col("product_id").isNull() | (F.length("product_id") == 0) | F.col("_has_product").isNull(), "invalid_product")
    .when(F.col("quantity").isNull() | (F.col("quantity") <= 0), "invalid_quantity")
    .when(F.col("unit_price").isNull() | (F.col("unit_price") <= 0), "invalid_price")
    .otherwise("invalid_identifier_or_status"))
quarantine = (joined.filter(invalid).withColumn("reject_reason", reason)
              .drop("_has_customer", "_has_product"))
clean = (joined.filter(~invalid).drop("_has_customer", "_has_product")
         .withColumn("line_amount", F.round(F.col("quantity") * F.col("unit_price"), 2)))
for name,frame in [("silver_customers", customers), ("silver_products", products),
                   ("silver_orders", clean), ("silver_orders_quarantine", quarantine)]:
    frame.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(name)
    print(name, frame.count())
assert clean.groupBy("order_id").count().filter("count > 1").count() == 0
